# Bravos Trade Status Viewer

Shows all Bravos trade alerts with parsed fields, order status, and whether the position is currently open in IBKR (paper account).

**Requires:** Cloud SQL Auth Proxy running on 127.0.0.1:5432, `BRAVOS_DB_PASSWORD` set in environment.

In [ ]:
import os
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host=os.environ.get('BRAVOS_DB_HOST', '127.0.0.1'),
    port=int(os.environ.get('BRAVOS_DB_PORT', '5432')),
    dbname=os.environ.get('BRAVOS_DB_NAME', 'bravos_trading'),
    user=os.environ.get('BRAVOS_DB_USER', 'bravos'),
    password=os.environ.get('BRAVOS_DB_PASSWORD', ''),
)
print('Connected to bravos_trading')

In [ ]:
query = """
WITH latest_gate AS (
    SELECT DISTINCT ON (signal_id)
        signal_id, gate_passed, checked_at
    FROM risk_gate_log
    ORDER BY signal_id, checked_at DESC
),
latest_order AS (
    SELECT DISTINCT ON (signal_id)
        signal_id, status AS order_status, fill_price, quantity AS order_qty
    FROM orders
    ORDER BY signal_id, created_at DESC
),
open_lots AS (
    -- Internal open lots per ticker
    SELECT ticker, SUM(quantity) AS open_qty_internal
    FROM position_lots
    WHERE lot_closed_at IS NULL
    GROUP BY ticker
),
ibkr_snapshot AS (
    -- Most recent broker position snapshot per ticker
    SELECT DISTINCT ON (ticker)
        ticker, position AS ibkr_qty, avg_cost AS ibkr_avg_cost, snapshot_at
    FROM broker_positions_snapshot
    ORDER BY ticker, snapshot_at DESC
)
SELECT
    s.id                                            AS signal_id,
    s.created_at AT TIME ZONE 'UTC'                 AS alert_date,
    s.ticker,
    s.action_type,
    s.weight_from,
    s.weight_to,
    s.reference_price,
    s.confidence,
    s.post_title,
    s.post_url,
    CASE
        WHEN lo.order_status IS NULL AND lg.gate_passed IS NULL THEN 'not_executed'
        WHEN lg.gate_passed = FALSE                             THEN 'risk_blocked'
        WHEN lo.order_status IS NULL                           THEN 'no_order'
        ELSE lo.order_status
    END                                             AS order_status,
    lo.order_qty,
    lo.fill_price,
    ol.open_qty_internal,
    ib.ibkr_qty,
    ib.ibkr_avg_cost,
    ib.snapshot_at                                  AS ibkr_snapshot_at,
    CASE
        WHEN ib.ibkr_qty IS NOT NULL AND ib.ibkr_qty != 0 THEN 'OPEN'
        WHEN ib.ibkr_qty = 0                               THEN 'CLOSED'
        ELSE 'UNKNOWN'
    END                                             AS ibkr_position_status
FROM signals s
LEFT JOIN latest_gate   lg ON lg.signal_id = s.id
LEFT JOIN latest_order  lo ON lo.signal_id = s.id
LEFT JOIN open_lots     ol ON ol.ticker    = s.ticker
LEFT JOIN ibkr_snapshot ib ON ib.ticker    = s.ticker
WHERE s.post_title != 'test'
  AND s.ticker NOT IN ('TEST', 'BADTKR')
  AND s.ticker IS NOT NULL
ORDER BY s.created_at DESC;
"""

df = pd.read_sql(query, conn)
df['alert_date'] = pd.to_datetime(df['alert_date']).dt.strftime('%Y-%m-%d %H:%M UTC')
print(f'{len(df)} signals loaded')
df.head()

## All trade alerts — most recent first

In [ ]:
display_cols = [
    'alert_date', 'ticker', 'action_type', 'weight_from', 'weight_to',
    'reference_price', 'confidence', 'order_status', 'fill_price',
    'ibkr_position_status', 'ibkr_qty', 'ibkr_avg_cost', 'post_title'
]

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 200)
df[display_cols]

## Currently open positions in IBKR

In [ ]:
open_pos = df[df['ibkr_position_status'] == 'OPEN'].drop_duplicates('ticker')
print(f'{len(open_pos)} tickers with open IBKR positions')
open_pos[['ticker', 'ibkr_qty', 'ibkr_avg_cost', 'ibkr_snapshot_at']].sort_values('ticker')

## Signals by order status

In [ ]:
df.groupby('order_status').size().rename('count').to_frame()

## High-confidence signals only

In [ ]:
high = df[df['confidence'] == 'high']
print(f'{len(high)} high-confidence signals')
high[display_cols]